# IRUFormer-GR Performance Evaluation
This notebook evaluates the performance of the **IRUFormer-GR** model on the test dataset of 57,458 images.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# Add the project root to sys.path to allow absolute imports
sys.path.append(str(Path(os.getcwd()).parent))

from src.utils.metrics import combined_denoising_loss, psnr_metric, ssim_metric, charbonnier_loss, ssim_loss
from src.utils.test_utils import get_tif_paths, select_random_test_images, load_image_normalized

print("Libraries and project modules loaded.")

In [ ]:
# Load the best model
model_path = "../results/models/best_model.keras"
if not os.path.exists(model_path):
    print(f"Warning: {model_path} not found. Please ensure your model is in the results/models/ directory.")
else:
    model = tf.keras.models.load_model(
        model_path,
        custom_objects={
            "combined_denoising_loss": combined_denoising_loss,
            "psnr_metric": psnr_metric,
            "ssim_metric": ssim_metric,
            "charbonnier_loss": charbonnier_loss,
            "ssim_loss": ssim_loss
        }
    )
    print(f"Model loaded from {model_path}")

In [ ]:
# Prepare Test Data
test_dir = Path("../data/test")
clean_paths = get_tif_paths(test_dir)

# Evaluate on the full set as per paper specifications
num_test_images = 57458
selected_files = select_random_test_images(clean_paths, num_images=num_test_images, seed=0)

psnr_values = []
ssim_values = []

print(f"Starting evaluation on {len(selected_files)} images...")

for i, clean_path_obj in enumerate(selected_files):
    clean_path = str(clean_path_obj)
    noisy_path = clean_path.replace("test", "test_gaussian_noise")
    
    if not Path(noisy_path).exists():
        continue

    x_noisy = load_image_normalized(noisy_path)
    y_true = load_image_normalized(clean_path)

    x_noisy_batch = np.expand_dims(x_noisy, axis=0)
    y_true_batch = np.expand_dims(y_true, axis=0)

    y_pred = model.predict(x_noisy_batch, verbose=0)
    y_pred = np.clip(y_pred, 0.0, 1.0)

    psnr = tf.image.psnr(y_true_batch, y_pred, max_val=1.0)
    ssim = tf.image.ssim(y_true_batch, y_pred, max_val=1.0)

    psnr_values.append(float(tf.reduce_mean(psnr).numpy()))
    ssim_values.append(float(tf.reduce_mean(ssim).numpy()))
    
    if (i + 1) % 1000 == 0:
        print(f"Processed {i + 1}/{len(selected_files)} images...")

average_psnr = np.mean(psnr_values)
average_ssim = np.mean(ssim_values)

print("\n" + "="*30)
print("FINAL TEST RESULTS")
print("="*30)
print(f"Average PSNR: {average_psnr:.4f} dB")
print(f"Average SSIM: {average_ssim:.4f}")